# MedVQA — Baseline Comparison
## Zero-shot and Text-only Baselines

Before fine-tuning, we establish baselines:
1. **CLIP ViT-L/14 zero-shot**: Binary classification on yes/no questions
2. **Mistral-7B text-only**: Image caption as text, no vision input
3. **Random baseline**: Random guessing for comparison

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor, AutoModelForCausalLM, AutoTokenizer

# Add project root
sys.path.insert(0, str(Path.cwd().parent))
from src.data.loader import VQARADDataset
from src.evaluation.metrics import compute_closed_ended_accuracy, compute_bleu, compute_rouge_l

print('Libraries loaded successfully')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# Load a small test subset
from src.data.preprocessor import MedicalImagePreprocessor, TextPreprocessor

image_processor = MedicalImagePreprocessor(image_size=224)
text_processor = TextPreprocessor()

test_dataset = VQARADDataset(
    data_dir='../data/raw/vqa_rad',
    split='test',
    image_transform=image_processor,
    text_preprocessor=text_processor,
)

print(f'Test dataset size: {len(test_dataset)}')

# Separate yes/no and open-ended
yesno_indices = [i for i in range(len(test_dataset)) 
                 if test_dataset.samples[i].get('question_type', '').lower() in ['yes/no', 'binary']]
open_indices = [i for i in range(len(test_dataset)) 
                if test_dataset.samples[i].get('question_type', '').lower() == 'open']

print(f'Yes/No questions: {len(yesno_indices)}')
print(f'Open-ended questions: {len(open_indices)}')

In [ ]:
# Baseline 1: Random guessing
print('=' * 50)
print('BASELINE 1: Random Guessing')
print('=' * 50)

rng = np.random.default_rng(42)

# Yes/no: random 50/50
n_yesno = len(yesno_indices)
random_yesno_preds = rng.choice(['yes', 'no'], n_yesno)
random_yesno_refs = [test_dataset.samples[i]['answer'] for i in yesno_indices]

random_acc = compute_closed_ended_accuracy(random_yesno_preds.tolist(), random_yesno_refs)
print(f'Random yes/no accuracy: {random_acc["accuracy"]:.4f}')

# Open-ended: pick random answers from training set
random_open_preds = rng.choice(['yes', 'no', 'unknown', 'normal', 'abnormal'], len(open_indices))
random_open_refs = [test_dataset.samples[i]['answer'] for i in open_indices]
random_bleu = compute_bleu(random_open_preds.tolist(), random_open_refs)
print(f'Random BLEU-1: {random_bleu["bleu_1"]:.4f}')
print(f'Random BLEU-4: {random_bleu["bleu_4"]:.4f}')

In [ ]:
# Baseline 2: CLIP zero-shot (yes/no questions only)
print('=' * 50)
print('BASELINE 2: CLIP ViT-L/14 Zero-Shot')
print('=' * 50)

try:
    clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
    clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
    print('CLIP model loaded')
except Exception as e:
    print(f'Could not load CLIP: {e}')
    clip_model = None

if clip_model is not None:
    clip_correct = 0
    for idx in tqdm(yesno_indices[:50], desc='CLIP zero-shot'):
        sample = test_dataset.samples[idx]
        
        # Load image
        image = Image.open(sample['image_path']).convert('RGB')
        
        # Prepare text prompts
        text_prompts = [f'A medical image showing {sample["question"]} Answer: yes.',
                        f'A medical image showing {sample["question"]} Answer: no.']
        
        inputs = clip_processor(text=text_prompts, images=image, return_tensors='pt', padding=True).to(device)
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
            logits_per_image = outputs.logits_per_image  # image-to-text similarity
            
        # Higher logit = more similar
        pred_idx = logits_per_image.argmax().item()
        pred = 'yes' if pred_idx == 0 else 'no'
        
        if pred == sample['answer'].lower().strip():
            clip_correct += 1
    
    clip_acc = clip_correct / min(50, len(yesno_indices))
    print(f'CLIP zero-shot yes/no accuracy: {clip_acc:.4f}')
else:
    print('Skipping CLIP baseline (model not available)')

In [ ]:
# Baseline 3: Mistral-7B text-only (no vision input)
print('=' * 50)
print('BASELINE 3: Mistral-7B Text-Only')
print('=' * 50)

try:
    mistral_tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.3')
    mistral_model = AutoModelForCausalLM.from_pretrained(
        'mistralai/Mistral-7B-Instruct-v0.3',
        torch_dtype=torch.float16,
        device_map='auto',
    )
    print('Mistral-7B loaded')
except Exception as e:
    print(f'Could not load Mistral: {e}')
    mistral_model = None

if mistral_model is not None:
    mistral_preds = []
    mistral_refs = []
    
    for idx in tqdm(range(min(30, len(test_dataset))), desc='Mistral text-only'):
        sample = test_dataset.samples[idx]
        question = sample['question']
        
        # Prompt without image (text-only baseline)
        prompt = f'[INST] Answer the following medical question based on general knowledge: {question} [/INST]'
        inputs = mistral_tokenizer(prompt, return_tensors='pt').to(device)
        
        with torch.no_grad():
            outputs = mistral_model.generate(
                **inputs,
                max_new_tokens=32,
                temperature=0.3,
                do_sample=True,
            )
        
        pred = mistral_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        mistral_preds.append(pred)
        mistral_refs.append(sample['answer'])
    
    mistral_bleu = compute_bleu(mistral_preds, mistral_refs)
    print(f'Mistral text-only BLEU-1: {mistral_bleu["bleu_1"]:.4f}')
    print(f'Mistral text-only BLEU-4: {mistral_bleu["bleu_4"]:.4f}')
    
    # Show some predictions
    print('\nSample predictions:')
    for i in range(5):
        print(f'  Q: {test_dataset.samples[i]["question"]}')
        print(f'  GT: {test_dataset.samples[i]["answer"]}')
        print(f'  Pred: {mistral_preds[i]}')
        print()
else:
    print('Skipping Mistral text-only baseline')

In [ ]:
# Baseline Comparison Summary
print('=' * 60)
print('BASELINE COMPARISON SUMMARY')
print('=' * 60)
print(f'{"Baseline":<30} {"Yes/No Acc":<15} {"BLEU-1":<10} {"BLEU-4":<10}')
print('-' * 65)
print(f'{"Random Guessing":<30} {"50.00%":<15} {"N/A":<10} {"N/A":<10}')
print(f'{"CLIP ViT-L/14 Zero-Shot":<30} {"TBD":<15} {"N/A":<10} {"N/A":<10}')
print(f'{"Mistral-7B Text-Only":<30} {"TBD":<15} {"TBD":<10} {"TBD":<10}')
print(f'{"MedVQA (Ours)":<30} {"TBD":<15} {"TBD":<10} {"TBD":<10}')
print('=' * 60)
print('\nAfter full training, update this table with actual results.')